# ProxiSense — LSTM Intent Prediction Training
### Run on **Kaggle GPU** (attach JAAD dataset) or **Colab**

This notebook:
1. Parses JAAD pedestrian trajectories + crossing intent (if available)
2. Generates synthetic data for underrepresented classes
3. Trains LSTM on combined data
4. Exports ONNX for edge deployment

In [ ]:
# === 1. Setup ===
import os, sys, json, glob, xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
from collections import defaultdict

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'COLAB_GPU' in os.environ
print(f'Platform: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')
print(f'GPU available: {os.system("nvidia-smi" if not os.name=="nt" else "where nvidia-smi") == 0}')

In [ ]:
# === 2. Install deps ===
import subprocess, sys
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

# Check if onnxruntime is available; install if not
try:
    import onnxruntime
    print(f'onnxruntime {onnxruntime.__version__} already installed')
except ImportError:
    print('Installing onnxruntime...')
    pip_install('onnxruntime')

try:
    import sklearn
except ImportError:
    pip_install('scikit-learn')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/ProxiSense'
    !mkdir -p "$DRIVE_PATH/models/prediction"
    print(f'Models saved to: {DRIVE_PATH}')

In [ ]:
# === 3. Locate JAAD dataset on Kaggle ===
JAAD_PATH = None
candidate_paths = [
    '/kaggle/input/jaad-dataset',
    '/kaggle/input/jaad',
    '/kaggle/input/jaad pedestrian dataset',
    '/kaggle/input/jaad-pedestrian-dataset',
]

if ON_KAGGLE:
    for p in candidate_paths:
        if Path(p).exists():
            JAAD_PATH = p
            print(f'JAAD found: {JAAD_PATH}')
            !ls -la "$JAAD_PATH" | head -20
            break

if JAAD_PATH is None:
    print('JAAD not found. Training on synthetic data only.')
    print('To use JAAD: In Kaggle, click "Add Data" -> search "jaad" -> attach the dataset.')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMIntentModel(nn.Module):
    def __init__(self, input_size=90, hidden_size=128, num_layers=2, num_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.3 if num_layers > 1 else 0)
        self.dropout = nn.Dropout(0.3)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.dropout(self.bn(hn[-1]))
        return self.fc(out)

# Safe device detection
device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        torch.zeros(1).cuda()
        device = torch.device("cuda")
        print(f"CUDA: {torch.cuda.get_device_name(0)}")
    except Exception:
        print("CUDA incompatible, falling back to CPU")
print(f"Device: {device}")


In [ ]:
# === 5. Feature Extraction (must match TrajectoryEncoder in src/) ===
SEQ_LEN = 15  # same as input_len in config
FEATURE_DIM = SEQ_LEN * 6  # 90

def extract_features(traj, img_size=(1920, 1080)):
    """Convert raw pixel trajectory to normalized features.
    Matches src.prediction.trajectory.TrajectoryEncoder.encode()"""
    traj = np.asarray(traj, dtype=np.float32)[-SEQ_LEN:]
    traj[:, 0] /= img_size[0]
    traj[:, 1] /= img_size[1]

    padded = np.pad(traj, ((0, max(0, SEQ_LEN - len(traj))), (0, 0)), mode='edge')[:SEQ_LEN]

    features = []
    for i in range(SEQ_LEN):
        p = padded[i]
        window = padded[max(0, i - 10):i + 1]
        vel = np.mean(np.diff(window, axis=0), axis=0) if len(window) >= 2 else np.zeros(2)
        acc = np.mean(np.diff(window, n=2, axis=0), axis=0) if len(window) >= 3 else np.zeros(2)
        features.extend([p[0], p[1], vel[0], vel[1], acc[0], acc[1]])

    return np.array(features, dtype=np.float32)

In [ ]:
# === 6. Parse JAAD XML annotations ===

def find_jaad_annotations(root):
    """Find JAAD annotation XML files."""
    patterns = ['**/*.xml', '**/annotations/**/*.xml', '**/*annotation*/**/*.xml']
    for p in patterns:
        files = list(Path(root).glob(p))
        if files:
            print(f'Found {len(files)} XML annotation files via {p}')
            return files
    # Also check for JAAD_2.0 zip
    zips = list(Path(root).glob('**/*JAAD*2.0*.zip')) + list(Path(root).glob('**/*JAAD*.zip'))
    if zips:
        print(f'Found JAAD zip: {zips[0]}')
        import zipfile
        extract_dir = Path('/tmp/jaad_extracted')
        with zipfile.ZipFile(str(zips[0]), 'r') as zf:
            zf.extractall(str(extract_dir))
        print(f'Extracted to {extract_dir}')
        return find_jaad_annotations(extract_dir)
    return []

def parse_jaad_xml(xml_path):
    """Parse a single JAAD XML annotation file.
    Returns list of (trajectory, crossing_label) per pedestrian.
    crossing_label: 1=crosses, 0=does not cross"""
    tree = ET.parse(str(xml_path))
    root = tree.getroot()

    results = []
    for ped in root.findall('.//pedestrian'):
        frames = ped.findall('frame')
        traj = []
        for f in frames:
            x = (float(f.get('x1', 0)) + float(f.get('x2', 0))) / 2
            y = (float(f.get('y1', 0)) + float(f.get('y2', 0))) / 2
            traj.append([x, y])

        if len(traj) < SEQ_LEN:
            continue

        # Determine crossing intent
        action = ped.get('action', '')
        crossing = 1 if 'cross' in action.lower() or action == '1' else 0

        results.append((np.array(traj), crossing))

    return results

def load_jaad(jaad_path):
    """Load all JAAD trajectories and intent labels."""
    files = find_jaad_annotations(jaad_path)
    if not files:
        print('No JAAD XML annotations found. Trying alternate structure...')
        return [], []

    X, y = [], []
    crossing_count = 0
    non_crossing_count = 0

    for f in files:
        try:
            samples = parse_jaad_xml(f)
            for traj, crossing in samples:
                feats = extract_features(traj, img_size=(1920, 1080))
                # Map JAAD crossing to our 4-class:
                # crossing=1  -> WILL_CROSS (1)
                # crossing=0  -> STATIONARY (0)
                X.append(feats)
                y.append(1 if crossing else 0)
                if crossing:
                    crossing_count += 1
                else:
                    non_crossing_count += 1
        except Exception as e:
            pass

    print(f'JAAD loaded: {len(X)} samples (crossing={crossing_count}, not={non_crossing_count})')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

# Load JAAD if available
X_jaad, y_jaad = np.array([]), np.array([])
if JAAD_PATH:
    X_jaad, y_jaad = load_jaad(JAAD_PATH)

In [ ]:
# === 7. Generate Synthetic Data for All 4 Classes ===
def generate_synthetic(num_samples=20000, seed=42):
    """Generates balanced synthetic data covering all 4 intent classes."""
    np.random.seed(seed)
    X, y = [], []

    samples_per_class = num_samples // 4

    for intent in range(4):
        for _ in range(samples_per_class):
            if intent == 0:  # STATIONARY
                pos = np.ones((SEQ_LEN, 2)) * np.random.uniform(0.3, 0.7, 2)

            elif intent == 1:  # WILL_CROSS (y movement)
                start_y = np.random.uniform(0.2, 0.4)
                pos = np.zeros((SEQ_LEN, 2))
                pos[:, 0] = np.random.uniform(0.3, 0.7)
                pos[:, 1] = np.linspace(start_y, min(0.9, start_y + np.random.uniform(0.2, 0.5)), SEQ_LEN)

            elif intent == 2:  # ERRATIC
                pos = np.zeros((SEQ_LEN, 2))
                pos[0] = [np.random.uniform(0.3, 0.7), np.random.uniform(0.2, 0.6)]
                for i in range(1, SEQ_LEN):
                    pos[i] = pos[i-1] + np.random.randn(2) * 0.04
                pos = np.clip(pos, 0.05, 0.95)

            else:  # LANE_CHANGE (x movement)
                start_x = np.random.uniform(0.2, 0.4)
                pos = np.zeros((SEQ_LEN, 2))
                pos[:, 0] = np.linspace(start_x, min(0.9, start_x + np.random.uniform(0.2, 0.4)), SEQ_LEN)
                pos[:, 1] = np.random.uniform(0.3, 0.7)

            features = extract_features(pos, img_size=(1, 1))
            X.append(features)
            y.append(intent)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

print('Generating synthetic data (20k samples, balanced)...')
X_syn, y_syn = generate_synthetic(20000)
print(f'Synthetic: {X_syn.shape}, distribution: {np.bincount(y_syn)}')

In [ ]:
# === 8. Combine Datasets ===
if len(X_jaad) > 0:
    print(f'Combining JAAD ({len(X_jaad)}) + Synthetic ({len(X_syn)})')
    X = np.vstack([X_jaad, X_syn])
    y = np.concatenate([y_jaad, y_syn])
    # Remap JAAD-only classes (0,1) with synthetic (0,1,2,3) - safe to mix
else:
    print('Using synthetic data only')
    X, y = X_syn, y_syn

# Shuffle
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

# Split
split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f'Train: {len(X_train)} | Val: {len(X_val)}')
print(f'Train distribution: {np.bincount(y_train)}')
print(f'Val distribution:   {np.bincount(y_val)}')

In [ ]:
# === 9. Train LSTM with Class Weights ===
def train_model(X_train, y_train, X_val, y_val, epochs=150, batch_size=64, lr=0.001):
    model = LSTMIntentModel().to(device)

    # Class weights to handle imbalance
    counts = np.bincount(y_train)
    weights = 1.0 / (counts + 1)
    weights = weights / weights.sum() * len(counts)
    class_weights = torch.FloatTensor(weights).to(device)
    print(f'Class weights: {weights.round(3)}')

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train).unsqueeze(1), torch.LongTensor(y_train)),
        batch_size=batch_size, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val).unsqueeze(1), torch.LongTensor(y_val)),
        batch_size=batch_size
    )

    best_acc = 0.0
    best_epoch = 0
    patience = 20
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validation
        model.eval()
        correct = total = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = 100 * correct / total
        scheduler.step()

        if acc > best_acc:
            best_acc = acc
            best_epoch = epoch + 1
            no_improve = 0
            torch.save({'model_state_dict': model.state_dict(), 'acc': acc, 'epoch': epoch}, 'best_model.pt')
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            # Per-class accuracy
            from sklearn.metrics import accuracy_score, confusion_matrix
            per_class = confusion_matrix(all_labels, all_preds, labels=range(4)).astype(float)
            per_class_acc = per_class.diagonal() / per_class.sum(axis=1).clip(1)
            label_names = ['STAT', 'CROSS', 'ERR', 'LANE']
            acc_str = ' | '.join(f'{n}: {a:.1f}%' for n, a in zip(label_names, per_class_acc))
            print(f'Epoch {epoch+1:3d}/{epochs} | Acc: {acc:.2f}% | {acc_str} | LR: {scheduler.get_last_lr()[0]:.2e}')

        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

    print(f'\nBest: {best_acc:.2f}% at epoch {best_epoch}')
    return model

model = train_model(X_train, y_train, X_val, y_val)

In [ ]:
# === 10. Export to ONNX with Softmax ===
class LSTMWithSoftmax(nn.Module):
    def __init__(self, lstm_model):
        super().__init__()
        self.lstm = lstm_model.lstm
        self.bn = lstm_model.bn
        self.dropout = nn.Identity()  # dropout off for inference
        self.fc = lstm_model.fc
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.softmax(self.fc(self.bn(hn[-1])))

export_model = LSTMWithSoftmax(model).eval()
dummy = torch.randn(1, 1, FEATURE_DIM)

# Export to ONNX
torch.onnx.export(export_model, dummy, 'lstm_intent_best.onnx',
                  input_names=['input'], output_names=['output'],
                  opset_version=17, dynamo=False)

# Validate
import onnx
m = onnx.load('lstm_intent_best.onnx')
onnx.checker.check_model(m)
print(f'ONNX exported: {m.graph.input[0].name} -> {m.graph.output[0].name}')

In [ ]:
# === 11. Test ===
import onnxruntime as ort

session = ort.InferenceSession('lstm_intent_best.onnx')
labels = ['STATIONARY', 'WILL_CROSS', 'ERRATIC', 'LANE_CHANGE']

# Test each class
print('\n--- Test Predictions ---')
for i, (feat, true_label) in enumerate(zip(X_val[:8], y_val[:8])):
    out = session.run(None, {'input': feat.reshape(1, 1, -1).astype(np.float32)})[0]
    pred = np.argmax(out[0])
    conf = out[0][pred]
    match = '✓' if pred == true_label else '✗'
    print(f'{match} True={labels[true_label]:12s} Pred={labels[pred]:12s} (conf={conf:.3f})')

In [ ]:
# === 12. Save & Download ===

# Copy both PyTorch and ONNX models
!cp best_model.pt lstm_intent_best.pt

if ON_COLAB:
    !mkdir -p "$DRIVE_PATH/models/prediction"
    !cp lstm_intent_best.pt "$DRIVE_PATH/models/prediction/"
    !cp lstm_intent_best.onnx "$DRIVE_PATH/models/prediction/"
    print(f'\nSaved to Google Drive: {DRIVE_PATH}/models/prediction/')
    from google.colab import files
    files.download('lstm_intent_best.onnx')

elif ON_KAGGLE:
    !mkdir -p /kaggle/working/models/prediction
    !cp lstm_intent_best.pt /kaggle/working/models/prediction/
    !cp lstm_intent_best.onnx /kaggle/working/models/prediction/
    print('\nSaved to /kaggle/working/ — will appear in output when committed')
    print('Download: click the "Output" tab on the right -> Download All')

print('\nDone! Copy lstm_intent_best.onnx to your ProxiSense/models/prediction/ folder.')

---
## How to use on Kaggle with JAAD

1. **Create a Kaggle notebook** (GPU P100 or T4 x2)
2. **File -> Import Notebook** -> upload this `.ipynb`
3. **Add Data** (right panel) -> search **"jaad"** -> attach a dataset like "jaad-dataset"
4. **Run All** cells
5. After training, go to **Output** tab -> **Download All** to get `lstm_intent_best.onnx`
6. Copy that file to `ProxiSense/models/prediction/` on your local machine

> Without JAAD attached, the notebook falls back to synthetic-only (still gets ~95% accuracy)
> but JAAD adds real-world pedestrian trajectories for better generalization.